# Please-YE 재현 테스트 (Colab, CPU)

심사위원 입장에서 **README.md만 보고 처음부터** 재현되는지 확인하는 노트북입니다.
GPU가 필요 없으니 런타임 유형은 **CPU**로 두세요 (런타임 → 런타임 유형 변경).

순서: 레포 받기 → 패키지 설치 → KITTI 다운로드(약 12 GB) → 압축 해제 → 평가기 → Release 모델 → `run_benchmark.py` → **GitHub에 올라간 결과와 자동 비교**

예상 시간 40~70분 (대부분 KITTI 다운로드와 CPU 예측). 위에서부터 한 셀씩 실행하면 됩니다.
막힌 셀이 있으면 그 지점이 README에서 고칠 곳입니다. 셀 번호와 오류 메시지를 기록해 주세요.

## 1. 레포 받기 (README 1절)

In [ ]:
%cd /content
!rm -rf Please-YE
!git clone https://github.com/youngeun05/Please-YE.git
%cd /content/Please-YE
!git log -1 --format='%h %ad %s' --date=iso
!df -h /content | tail -1

## 2. 패키지 설치 (README 1절, Linux)

코랩에는 GPU용 torch가 미리 깔려 있지만, README와 똑같이 CPU 버전을 설치합니다. 경고는 무시해도 됩니다.

In [ ]:
!python -m pip install -q --upgrade pip
!python -m pip install -q torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cpu
!python -m pip install -q -r requirements-tested.txt -r requirements-onnx.txt
!python -c "import torch, ultralytics, onnxruntime, numpy; print('torch', torch.__version__, '| ultralytics', ultralytics.__version__, '| ort', onnxruntime.__version__, '| numpy', numpy.__version__)"

## 3. KITTI 다운로드 (README 2절)

라벨(5 MB) → 이미지(약 12 GB) 순서입니다. 끊기면 셀을 다시 실행하면 이어받습니다.

In [ ]:
!mkdir -p /content/downloads
!curl -L -C - -o /content/downloads/data_object_label_2.zip https://s3.eu-central-1.amazonaws.com/avg-kitti/data_object_label_2.zip
!curl -L -C - -o /content/downloads/data_object_image_2.zip https://s3.eu-central-1.amazonaws.com/avg-kitti/data_object_image_2.zip
!ls -la /content/downloads

## 4. 압축 해제 + 검사 (README 2절)

ZIP 전체 CRC를 검사한 뒤 학습용 7,481쌍만 꺼냅니다. 수 분 걸립니다. 끝나면 디스크를 아끼려고 이미지 ZIP을 지웁니다.

In [ ]:
!python scripts/prepare_kitti.py --image-zip /content/downloads/data_object_image_2.zip --label-zip /content/downloads/data_object_label_2.zip --eval-split eval_val.txt --output-root /content/datasets/KITTI
!ls /content/datasets/KITTI/training/image_2 | wc -l
!ls /content/datasets/KITTI/training/label_2 | wc -l
!rm -f /content/downloads/data_object_image_2.zip

## 5. 평가기 (README 3절, Linux)

In [ ]:
!bash scripts/setup_evaluator.sh

## 6. Release 모델 받기 (README 4절)

받은 파일의 SHA-256을 레포에 커밋된 `models/manifest.json`과 대조합니다.

In [ ]:
!curl -L -O https://github.com/youngeun05/Please-YE/releases/download/prelim-v1/please-ye-models.zip
!unzip -o -q please-ye-models.zip
!ls -la models

import hashlib, json, pathlib
manifest = json.load(open('models/manifest.json', encoding='utf-8'))
for e in manifest['models']:
    sha = hashlib.sha256(pathlib.Path('models', e['file']).read_bytes()).hexdigest()
    print(f"{e['file']:<36} {'OK' if sha == e['sha256'] else 'MISMATCH'}  {sha[:12]}")

## 7. 벤치마크 실행 (README 5절)

심사위원과 같은 명령입니다. 정확도는 PC와 무관하게 같아야 하고, 속도는 코랩 CPU라 PC00 값과 다릅니다 (정상).

In [ ]:
!python scripts/run_benchmark.py --kitti-root /content/datasets/KITTI --split eval_val --confirm-final

## 8. 기대값과 자동 비교 (CPU 종류에 맞춰)

ONNX Runtime의 INT8 결과는 **CPU에 VNNI 명령어가 있느냐**에 따라 달라집니다 (VNNI 없음: 곱셈 중간값이 16비트에서 잘림).
FP32 baseline은 어떤 CPU에서나 같습니다. 이 셀은 CPU를 먼저 검사한 뒤 `results/eval_val_expected_by_cpu.json`에서
그 CPU 종류의 기대값을 골라 비교합니다. 기준: 차이 0.01점 이내.


In [ ]:
import json, subprocess, sys
probe = json.loads(subprocess.run([sys.executable, 'scripts/int8_probe.py'], capture_output=True, text=True).stdout.split('\n\n')[0])
path = 'no_vnni' if probe['saturates'] else 'vnni'
print(f"이 CPU의 INT8 경로: {probe['int8_path']}  (검사값 {probe['got']:,} / 정확한 값 {probe['expected_exact']:,})")
expected = json.load(open('results/eval_val_expected_by_cpu.json', encoding='utf-8'))
ref = expected['values'][path]
print('비교 기준:', ref['source'], '\n')
mine = json.load(open('runs/benchmark/eval_val/summary.json', encoding='utf-8'))['accuracy']
ok = True
for label in ('baseline', 'optimized'):
    a = mine[label]
    for name in ('mean', 'Car', 'Pedestrian', 'Cyclist'):
        x = a['mean_ap40_moderate'] if name == 'mean' else a['class_ap40_moderate'][name]
        y = ref[label][name]
        good = abs(x - y) <= expected['tolerance']
        ok &= good
        print(f"{label:<10} {name:<11} colab {x:7.3f}  기대값 {y:7.3f}  diff {x - y:+.4f}  {'OK' if good else 'DIFFERENT'}")
print()
print('재현 성공: 이 CPU 종류의 기대값과 정확도가 일치합니다.' if ok else '재현 실패: 차이가 0.01점을 넘습니다. 위 표를 공유해 주세요.')


In [ ]:
print(open('runs/benchmark/eval_val/summary.md', encoding='utf-8').read())